# Research - VoxelGrid carving with Open3D

In [ ]:
import numpy as np
import open3d as o3d

from os import environ

from plant3dvision.visu import plotly_pointcloud_data
from plant3dvision.utils import locate_task_filesets
from plant3dvision.camera import get_camera_kwargs_from_images_metadata
from plant3dvision.camera import colmap_params_from_kwargs
from plantdb.commons.fsdb.core import FSDB
from plantdb.commons.io import read_image

In [ ]:
db = FSDB(environ.get('ROMI_DB', "/data/ROMI/Romi_Alexis/analyse_jo"))
db.connect(unsafe=True)

In [ ]:
scan = db.get_scan('Col-0_E1_1', create=False)

In [ ]:
fileset_names = locate_task_filesets(scan, ["images", "Colmap", "Mask"])

In [ ]:
mask_fs = scan.get_fileset(fileset_names['Mask'])

In [ ]:
mask_files = mask_fs.get_files()

In [ ]:
colmap_fs = scan.get_fileset(fileset_names['Colmap'])

In [ ]:
bbox = colmap_fs.get_metadata('bounding_box')

In [ ]:
voxel_size = 1.0

In [ ]:
x_min, x_max = 300, 435
y_min, y_max = 300, 435
z_min, z_max = -300, 60

In [ ]:
# - Define the shape of the voxel array:
nx = float(int((x_max - x_min) / voxel_size) + 1)
ny = float(int((y_max - y_min) / voxel_size) + 1)
nz = float(int((z_max - z_min) / voxel_size) + 1)

In [ ]:
# - Defines the origin of the voxel array:
origin = np.array([x_min, y_min, z_min])

In [ ]:
vx_grid = o3d.geometry.VoxelGrid()
vx_grid.create_dense(origin=origin, color=np.array([1., 1., 1.]),
                     voxel_size=voxel_size, width=nx, height=ny, depth=nz)

In [ ]:
for mask_file in mask_files:
    mask = o3d.geometry.Image(read_image(mask_file))
    mask_md = mask_file.get_metadata()
    cam_md = mask_md["colmap_camera"]['camera_model']
    intrinsic_md = get_camera_kwargs_from_images_metadata(mask_file)
    if intrinsic_md['model'].lower() != "opencv":
        opencv_cam = colmap_params_from_kwargs(**intrinsic_md)
        intrinsic_md = dict(zip(['fx', 'fy', 'cx', 'cy', 'k1', 'k2', 'p1', 'p2'], opencv_cam))
    cam_intrinsic = o3d.camera.PinholeCameraIntrinsic()
    cam_intrinsic.set_intrinsics(width=cam_md['width'], height=cam_md['height'],
                                 fx=intrinsic_md['fx'], fy=intrinsic_md['fy'],
                                 cx=intrinsic_md['cx'], cy=intrinsic_md['cy'])
    cam_params = o3d.camera.PinholeCameraParameters()
    cam_params.intrinsic = cam_intrinsic
    cam_extrinsic = np.array([[0., 0., 0., 0.], [0., 0., 0., 0.], [0., 0., 0., 0.], [0., 0., 0., 1.]])
    rotmat = np.array(mask_md["colmap_camera"]["rotmat"])
    tvec = np.array(mask_md["colmap_camera"]["tvec"])
    cam_extrinsic[:3, :3] = rotmat.T
    cam_extrinsic[:3, 3] = -rotmat.T @ tvec
    cam_params.extrinsic = np.linalg.inv(cam_extrinsic)
    vx_grid.carve_silhouette(mask, cam_params)

In [ ]:
Rt = np.array(mask_md["colmap_camera"]["rotmat"]).T
t = np.array(mask_md["colmap_camera"]["tvec"])

In [ ]:
-Rt@t

In [ ]:
vx_grid.dimension

In [ ]:
cam_params.intrinsic.intrinsic_matrix

In [ ]:
cam_params.extrinsic

In [ ]:
mask = o3d.geometry.Image(read_image(mask_files[0]))

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(np.asarray(mask), cmap='gray')

In [ ]:
db.disconnect()